# YOLO vs SegFormer Pipeline Comparison

This notebook runs both YOLO and SegFormer on satellite tiles from multiple Atlanta locations and displays **side-by-side** comparisons (Original | YOLO | SegFormer) for up to 25 parking lots.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
import os

from parksight.fetch import get_parking_data, get_satellite_tile
from parksight.segment import ParkingSegmenter
from parksight import is_structure
from yolo.detect import YOLOParkingDetector
from parksight.viz import create_parking_map

# Load YOLO
weights_path = "models/yolo26n_run1.pt" if os.path.exists("models/yolo26n_run1.pt") else (
    "models/best.pt" if os.path.exists("models/best.pt") else "runs/parksight_yolo/weights/best.pt"
)
detector = YOLOParkingDetector(weights_path)

# Load SegFormer
segformer_path = "models/best_model"
segmenter = None
if os.path.exists(segformer_path):
    segmenter = ParkingSegmenter(segformer_path)
    print(f"SegFormer loaded from {segformer_path}")
else:
    print(f"SegFormer not found at {segformer_path}, will show YOLO only")

MAX_EXAMPLES = 25

## Process multiple locations

We query 8 different Atlanta-area locations to gather up to 25 surface-lot tiles.

In [ ]:
locations = [
    "Georgia Tech campus, Atlanta, GA",
    "Lenox Square Mall, Atlanta, GA",
    "Midtown, Atlanta, GA",
    "Atlantic Station, Atlanta, GA",
    "Emory University, Atlanta, GA",
    "Piedmont Park, Atlanta, GA",
    "Hartsfield-Jackson Airport, Atlanta, GA",
    "Perimeter Mall, Dunwoody, GA",
]

all_tiles = []  # list of (name, img, yolo_ann, seg_ann, yolo_count, seg_count)

for loc in locations:
    if len(all_tiles) >= MAX_EXAMPLES:
        break
    print(f"\nProcessing: {loc}")
    try:
        gdf, (lat, lon) = get_parking_data(loc, dist=300)
    except Exception as e:
        print(f"  Skipped ({e})")
        continue
    gdf_3857 = gdf.to_crs(epsg=3857)

    for idx, row in gdf_3857.iterrows():
        if len(all_tiles) >= MAX_EXAMPLES:
            break
        geom = row.geometry
        tags = row.to_dict()
        if is_structure(tags) or geom.geom_type not in ("Polygon", "MultiPolygon"):
            continue

        img = get_satellite_tile(geom)

        # YOLO
        yolo_dets = detector.detect(img)
        yolo_count = detector.count_spots(img, geom, osm_tags=tags)
        yolo_ann = detector.annotate(img, yolo_dets)

        # SegFormer
        if segmenter is not None:
            seg_result = segmenter.count_spots(img)
            seg_count = seg_result.count
            seg_ann = segmenter.annotate(img, seg_result.mask)
        else:
            seg_count = "-"
            seg_ann = img.copy()

        name = tags.get("name", None)
        if not isinstance(name, str) or not name:
            name = f"Lot #{idx}"
        short_loc = loc.split(",")[0]
        label = f"{short_loc} / {name}"[:40]

        all_tiles.append((label, img, yolo_ann, seg_ann, yolo_count, seg_count))

print(f"\nCollected {len(all_tiles)} surface-lot tiles across {len(locations)} locations.")

## Side-by-side comparison: Original | YOLO | SegFormer

In [ ]:
for i, (label, img, yolo_ann, seg_ann, yolo_count, seg_count) in enumerate(all_tiles):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img)
    axes[0].set_title(f"[{i+1}] {label}", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(yolo_ann)
    axes[1].set_title(f"YOLO ({yolo_count} spots)", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(seg_ann)
    axes[2].set_title(f"SegFormer ({seg_count} spots)", fontsize=10)
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

## Summary table

In [ ]:
import pandas as pd

rows = []
for i, (label, _, _, _, yolo_count, seg_count) in enumerate(all_tiles):
    rows.append({"#": i + 1, "Location / Lot": label, "YOLO": yolo_count, "SegFormer": seg_count})

df = pd.DataFrame(rows)
display(df.style.hide(axis="index"))

yolo_total = sum(r["YOLO"] for r in rows if isinstance(r["YOLO"], (int, float)))
seg_total = sum(r["SegFormer"] for r in rows if isinstance(r["SegFormer"], (int, float)))
print(f"\nTotals  —  YOLO: {yolo_total}  |  SegFormer: {seg_total}")